<img src="../../../On-Site-Round/Help_BOBAI/figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (Burgas, Bulgarie), epreuve sur site](https://ioai-official.org/bulgaria-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2024/blob/main/fr/On-Site-Round/Help_BOBAI/Help_BOBAI.ipynb)

# Help BOBAI : davantage de classification dans une langue inconnue

<img src="../../../On-Site-Round/Help_BOBAI/figs/Help BOBAI Fig 1.png" width="700">

## Contexte
La derniere fois que Bob vous a contacte, il vous a demande de l'aider a construire un classifieur pour une nouvelle langue inconnue. La cliente, Amoira, a ete satisfaite de votre solution ; Bob a donc demande a son equipe de deployer le nouveau modele et, apres une forte optimisation ainsi que des tests unitaires soigneux, le service a ete mis en production et fonctionne correctement depuis.

## Tache

Ce matin meme, Amoira est revenue avec une demande : faire passer le nombre de classes prises en charge par le classifieur de 5 a 7. Et cela doit etre fait *aujourd'hui* !

Amoira a fourni des donnees etiquetees pour les nouvelles classes. Avec plus de temps, Bob pourrait simplement reutiliser votre solution precedente pour entrainer un nouveau modele sur l'union des anciennes et des nouvelles donnees, n'est-ce pas ? Le probleme est que le deploiement d'un nouveau modele est un processus complexe qui ne peut pas etre mene en une journee ; la solution doit donc etre construite entierement autour du modele deja deploye. Bob revient donc vers vous, puisque vous connaissez le mieux la tache.

De plus, les preoccupations de securite d'Amoira ont encore augmente avec l'ajout des nouvelles donnees. Elle a donc demande a Bob de ne publier le texte sous aucune forme : que se passerait-il si quelqu'un parvenait a le dechiffrer ? Bob vous fournit donc un encodage pre-calcule et mis en cache de toutes les donnees disponibles : les ensembles d'entrainement et de developpement deja utilises pour la classification a 5 classes, ainsi que les nouvelles donnees fournies par Amoira pour les 2 classes supplementaires. Cet encodage correspond a la sortie de la couche de pooling de mBERT ; il s'integre donc directement au classifieur deja entraine.

Votre objectif est de construire une solution de classification a 7 classes tout en respectant les contraintes suivantes :

* La solution peut utiliser le classifieur a 5 classes, mais elle ne peut ni modifier ses parametres ni ajouter de nouveaux parametres appris.
* Vous pouvez calculer des moyennes et des distances entre les encodages des donnees.
* La solution doit etre reproductible en moins d'une heure sur une carte GPU L4.
* Le classifieur doit pouvoir effectuer l'inference sur n'importe quel ensemble aleatoire de 500 echantillons en moins de 2 minutes sur une carte GPU L4.

## Livrables

Vous devez remettre :

* Du code fonctionnel permettant de reproduire et de tester votre meilleur modele.
  * Dans ce notebook Colab.
  * Reproduire votre meilleur modele signifie qu'en partant du classifieur de base, nous devons pouvoir atteindre votre meilleur modele final simplement en executant les cellules du notebook.
* Les predictions sur les donnees de test (publiees deux heures avant la fin de la competition).

**Vous devez absolument vous assurer que :**

(1) votre notebook est executable du debut a la fin

(2) le notebook contient tout le code necessaire pour reproduire votre modele

(3) il peut s'executer sur un GPU L4


## Jeu de donnees d'entrainement

In [ ]:
import torch

dataset = torch.load('./training_set/train-dev_dataset_with_labels.pt')

inputs = dataset[:,:,:-1]
labels = dataset[:, :, -1]


## Solution de reference

Vous trouverez ci-dessous une solution de reference tres naive : pour un vecteur d'entree donne, soit nous attribuons aleatoirement l'une des nouvelles etiquettes (5 et 6) avec une probabilite uniforme dans une classification a 7 classes, soit nous utilisons le classifieur de base pour produire une prediction.

Vous pouvez remplacer le code ci-dessous par votre propre solution.

In [ ]:
import torch
import random

class SevenWayClassifier():
  def __init__(self, ):
    base_clf = torch.nn.Linear(in_features=768, out_features=5, bias=True)
    base_clf.load_state_dict(torch.load("training_set/base_classifier.pth"))
    self.base_clf = base_clf

  def base_classification(self, input_vector):

    with torch.no_grad():
      logits = self.base_clf(input_vector)
      preds = torch.softmax(logits, 1)
      predicted_class = preds.argmax(dim=1).numpy()[0]

    return predicted_class

  def __call__(self, input_vector):

    random_class = random.choice([0,1,2,3,4,5,6])
    if random_class in [5,6]:
      predicted_class = random_class
    else:
      predicted_class = self.base_classification(input_vector)

    return predicted_class

clf = SevenWayClassifier()

## Inference et evaluation

In [ ]:
from sklearn.metrics import f1_score

def compute_f1(labels, predictions):
  return f1_score(labels, predictions, average='macro')

In [ ]:
from tqdm import tqdm

def inference(clf, input_vectors):
  predictions = []
  for sample in tqdm(input_vectors):
    predictions.append(clf(sample))
  return predictions

In [ ]:

predictions = inference(clf, inputs)

f1 = compute_f1(labels, predictions)
print('\nNaive solution F1', f1)

## Jeu de donnees de validation

In [ ]:
# Le leaderboard peut fonctionner ou non... S'il ne fonctionne pas, soyez indulgents. Nous essaierons de le remettre en route.

import pandas as pd
import numpy as np

def submission_to_csv(pred: np.ndarray, output_fpath: str = "submission.csv"):
    pred = np.array(pred).flatten()
    data_size = pred.size
    df = pd.DataFrame({
        "ID": np.arange(data_size),
        "class": pred
    })

    df.to_csv(output_fpath, index=False)

eval_inputs = torch.load('./Solution/validation_set/eval_dataset.pt')

eval_predictions = inference(clf, eval_inputs)

submission_to_csv(eval_predictions)

## Jeu de donnees de test

In [ ]:
# NE MODIFIEZ PAS CETTE CELLULE

# ce lien de telechargement ne fonctionnera que deux heures avant la fin de la competition
test_inputs = torch.load('Solution/test_set/test_dataset.pt')

split='test'

test_predictions = inference(clf, test_inputs)

with open('{}_predictions.txt'.format("Team Name"), 'w') as outfile:
  outfile.write('\n'.join([str(p) for p in test_predictions]))